In [1]:
import os

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

In [3]:
%pwd

'c:\\Users\\aabhi\\Desktop\\kidney_disease\\Kidney-Disease-Classification\\research'

In [4]:
os.chdir(PROJECT_ROOT)

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [7]:
import os
os.getcwd()

'c:\\Users\\aabhi\\Desktop\\kidney_disease\\Kidney-Disease-Classification'

In [8]:
import os
print(os.listdir())

['.git', '.github', '.gitignore', '.venv', 'artifacts', 'config', 'dvc.yaml', 'index.html', 'logs', 'main.py', 'params.yaml', 'README.md', 'requirements.txt', 'research', 'setup.py', 'src', 'template.py', 'templates']


In [9]:
import sys
print(sys.executable)


c:\Users\aabhi\AppData\Local\Programs\Python\Python310\python.exe


In [10]:
from pathlib import Path

CONFIG_FILE_PATH = Path("src/cnnClassifier/config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")

In [11]:
from cnnClassifier.entity.config_entity import DataIngestionConfig

In [12]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [14]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [21]:
import os
import zipfile
import urllib.request as request

from src.cnnClassifier import logger
from src.cnnClassifier.entity.config_entity import DataIngestionConfig


class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> str:
        """
        Fetch data from URL
        """

        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file

            os.makedirs("artifacts/data_ingestion", exist_ok=True)

            logger.info(
                f"Downloading data from {dataset_url} into file {zip_download_dir}"
            )

            filename, headers = request.urlretrieve(
                url=dataset_url,
                filename=zip_download_dir
            )

            logger.info(
                f"Downloaded data from {dataset_url} into file {zip_download_dir}"
            )

            return filename

        except Exception as e:
            logger.exception(e)
            raise

    def extract_zip_file(self):
        """
        Extracts the zip file into the data directory
        """

        try:
            unzip_path = self.config.unzip_dir

            os.makedirs(unzip_path, exist_ok=True)

            logger.info(f"Extracting zip file into {unzip_path}")

            with zipfile.ZipFile(
                self.config.local_data_file,
                "r"
            ) as zip_ref:

                zip_ref.extractall(unzip_path)

            logger.info("Zip file extracted successfully")

        except Exception as e:
            logger.exception(e)
            raise

In [22]:
try:
    config = ConfigurationManager()

    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)

    data_ingestion.download_file()

    data_ingestion.extract_zip_file()

except Exception:
    raise

[2026-05-08 22:40:34,165: INFO: common: YAML file: src\cnnClassifier\config\config.yaml loaded successfully]
[2026-05-08 22:40:34,167: INFO: common: YAML file: params.yaml loaded successfully]
[2026-05-08 22:40:34,168: INFO: common: Created directory at: artifacts]
[2026-05-08 22:40:34,170: INFO: common: Created directory at: artifacts/data_ingestion]
[2026-05-08 22:40:34,170: INFO: 4269190096: Downloading data from https://archive.ics.uci.edu/static/public/336/chronic+kidney+disease.zip into file artifacts/data_ingestion/data.zip]
[2026-05-08 22:40:37,208: INFO: 4269190096: Downloaded data from https://archive.ics.uci.edu/static/public/336/chronic+kidney+disease.zip into file artifacts/data_ingestion/data.zip]
[2026-05-08 22:40:37,208: INFO: 4269190096: Extracting zip file into artifacts/data_ingestion]
[2026-05-08 22:40:37,208: INFO: 4269190096: Zip file extracted successfully]
